# L18 Notes: CIFAR-10 Image Processing for ML

### Set up imports

In [ ]:
import pickle
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score

### Load data

In [ ]:
with open('cifar_data', 'rb') as file:
    cifar_data = pickle.load(file, encoding='latin1')

type(cifar_data)

In [ ]:
cifar_data.keys()

### Look at first image

In [ ]:
image_data = cifar_data['data']
label_data = cifar_data['labels']

image_data[0].shape

In [ ]:
# define a function to show an image and print a label
def show_image(image, label):
    # first convert the flattened image to 3d array
    img = image.reshape(3, 32, 32)

    # then, change the order of the dimensions
    img = img.transpose(1, 2, 0)

    # show the image
    plt.imshow(img)

    plt.show()

    # define list of labels
    label_names = ['airplane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

    # print the label
    print(label_names[label])

# test the function
show_image(image_data[1], label_data[1])

### Split the data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(image_data, label_data, random_state=42)

X_train.shape

### Define and train the model

In [ ]:
model = RandomForestClassifier(n_estimators=20, random_state=42)

model.fit(X_train, y_train)

### Make predictions and evaluate accuracy

In [ ]:
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)

print(f'accuracy: {acc * 100}%')

# Activity

Goal: Make a model that performs better on the test data

Steps: 
1. Train a gradient boosting classifer with default params and measure accuracy

2. Perform grid search (use cv=2 to speed things up) with the random forest
Parameters to test:
n_estimators: [10, 20, 100]
max_features: ['sqrt', 'log2']
max_samples: [2000, 5000, None]

3. Analyze the mistakes of the best fitting model
Check the first 5 errors. Show the image, the predicted label, and the true label

### Train gradient boosting classifier and measure accuracy

In [ ]:
# define the model
hgb_clf = HistGradientBoostingClassifier(random_state=42, max_iter=10, verbose=2)

# train the model
hgb_clf.fit(X_train, y_train)

# make test set predictions
y_pred_hgb = hgb_clf.predict(X_test)

# quantify accuracy
acc_hgb = accuracy_score(y_test, y_pred_hgb)

print(f'accuracy (gradient boosting): {acc_hgb * 100}%')

### Run grid search to improve random forest fitting
2. Perform grid search (use cv=2 to speed things up) with the random forest
Parameters to test:
n_estimators: [10, 20, 100]
max_features: ['sqrt', 'log2']
max_samples: [1000, 2000, None]


In [ ]:
# import the gridsearchCV
from sklearn.model_selection import GridSearchCV

# define the parameters to test
hyperparam_grid = {
    'n_estimators': [10, 20, 100],
    'max_features': ['sqrt', 'log2'],
    'max_samples': [1000, 2000, None]
}

# define the grid search
grid = GridSearchCV(estimator=RandomForestClassifier(), param_grid=hyperparam_grid, cv=2, verbose=2)

# run the grid search
grid.fit(X_train, y_train)

### Train a model with the best parameters and find the accuracy

In [ ]:
print(f'The best params are {grid.best_params_}')

best_model = grid.best_estimator_

y_pred_best = best_model.predict(X_test)

acc_best = accuracy_score(y_test, y_pred_best)

acc_best

### Look at the errors the best model is making
Loop over the y_pred_best and y_test together

check if there is a difference

if there is, then show the image, the true label, and the predicted label


In [ ]:
label_names = ['airplane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
index = 0

for true_label, pred_label in zip(y_test, y_pred_best):
    
    # check for error
    if true_label != pred_label:

        # get the label names
        true_label_name = label_names[true_label]
        pred_label_name = label_names[pred_label]

        # print the label names
        print(true_label_name, pred_label_name)
        show_image(X_test[index], y_test[index])

    index = index + 1